# FoldMason Toolkit

This notebook demonstrates the two FoldMason tools:

1. **`foldmason-msa`** — multiple structure alignment (remote default; tuneable local mode)
2. **`foldmason-score-msa`** — score an existing MSA with average + per-column LDDT (local-only)

Reference: Gilchrist et al., *Science* 2026 ([DOI](https://doi.org/10.1126/science.ads6733)).

In [1]:
import requests
from proto_tools.tools.structure_alignment import (
    FoldmasonMSAConfig, FoldmasonMSAInput, run_foldmason_msa,
    FoldmasonScoreMSAConfig, FoldmasonScoreMSAInput, run_foldmason_score_msa,
)

## Step 1: fetch a TIM-barrel structural family

We'll align three TIM-barrel triosephosphate isomerases from different organisms — a classic example of structurally similar enzymes spanning broad sequence divergence.

In [2]:
tim_pdbs = {}
for pdb_id in ("1TIM", "8TIM", "1TPF"):
    tim_pdbs[pdb_id] = requests.get(f"https://files.rcsb.org/download/{pdb_id}.pdb", timeout=30).text
structures = list(tim_pdbs.values())
ids = list(tim_pdbs.keys())
print(f"Fetched {len(structures)} TIM-barrel structures: {ids}")

Fetched 3 TIM-barrel structures: ['1TIM', '8TIM', '1TPF']


## Step 2: align with `foldmason-msa` (local)

Runs the FoldMason CLI in an isolated standalone env. We use local mode here so the next step (`foldmason-score-msa`, which is local-only) can resolve each MSA row to its structure — the public server's MSA naming differs from the local createdb naming for multi-chain inputs.

In [3]:
msa_out = run_foldmason_msa(
    FoldmasonMSAInput(structures=structures, structure_ids=ids),
    FoldmasonMSAConfig(search_mode="local", num_threads=2),
)
print(f"Aligned {msa_out.num_sequences} structures across {msa_out.alignment_length} columns")
print(f"Guide tree: {msa_out.newick_tree.strip()}")
print()
for line in msa_out.aa_msa_fasta.splitlines()[:6]:
    print(line[:100] + (" ..." if len(line) > 100 else ""))

Running run_foldmason_msa [00:00]

Aligned 6 structures across 251 columns
Guide tree: ((1TPF_A,1TPF_B),(1TIM_A,(1TIM_B,(8TIM_A,8TIM_B))));

>8TIM_A
-APRKFFVGGNWKMNGDKKSLGELIHTLNGAKLSADTEVVCGAPSIYLDFARQKL-DAKIGVAAQNCYKVPKGAFTGEISPAMIKDIGAAWVILGHSERR ...
>8TIM_B
-APRKFFVGGNWKMNGDKKSLGELIHTLNGAKLSADTEVVCGAPSIYLDFARQKL-DAKIGVAAQNCYKVPKGAFTGEISPAMIKDIGAAWVILGHSERR ...
>1TIM_B
-APRKFFVGGNWKMNGKRKSLGELIHTLDGAKLSADTEVVCGAPSIYLDFARQKL-DAKIGVAAQNCYKVPKGAFTGEISPAMIKDIGAAWVILGHSERR ...


## Step 3: score the alignment with `foldmason-score-msa` (local)

Computes per-column and average LDDT scores against the input structures. Useful as a quality metric or downstream design constraint. Local-only — `msa2lddt` is not exposed by the public server.

In [4]:
score_out = run_foldmason_score_msa(
    FoldmasonScoreMSAInput(structures=structures, structure_ids=ids, aa_msa_fasta=msa_out.aa_msa_fasta),
    FoldmasonScoreMSAConfig(num_threads=2),
)
print(f"Average MSA LDDT: {score_out.average_lddt:.3f}")
print(f"Columns considered: {score_out.columns_considered}/{score_out.alignment_length}")
print(f"First 10 column scores: {[round(s, 2) for s in score_out.column_scores[:10]]}")

Running run_foldmason_score_msa [00:00]

Average MSA LDDT: 0.923
Columns considered: 251/251
First 10 column scores: [0.0, 0.68, 0.77, 0.9, 0.93, 0.92, 0.94, 0.95, 0.95, 0.96]
